In [19]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from pysindy.feature_library import PolynomialLibrary, GeneralizedLibrary
from pysindy import SINDy
import pysindy as ps
from sklearn.metrics import r2_score

In [20]:
# Define the system of ODEs
def system(t, variables):
    x, y, mu = variables
    dxdt = mu * x + y - x * (x**2 + y**2)
    dydt = -x + mu * y - y * (x**2 + y**2)
    dmudt= 0
    return [dxdt, dydt, dmudt]

In [23]:
# Initial conditions
initial_conditions = [0.01, 0.01, 0.2]  # Initial values for x, y, and mu
alpha = 1  # Parameter alpha
threshold = 0.1

# Time span
t_span = (0, 20)
t_eval = np.arange(0, 20, 0.01)

# Solve the system
sol = solve_ivp(system, t_span, initial_conditions, t_eval=t_eval, dense_output=True)

# Extract the solution at the evaluated time points
x = sol.y.T

# Compute the true derivatives using the system function
true_derivatives = np.array([system(t, xi) for t, xi in zip(t_eval, x)])

# SINDy modeling with the parameterized feature library
optimizer = ps.STLSQ(threshold=threshold)
model = ps.SINDy(
    optimizer=optimizer,
    feature_library=PolynomialLibrary(degree=3),
    feature_names=['x', 'y', 'mu']
)

# Fit the SINDy model
model.fit(x, t=t_eval[1]-t_eval[0])
model.print()

# Predict the derivatives using the trained model
predicted_derivatives = model.predict(x)

# Extract the true and predicted derivatives for x and y
true_derivatives_xy = true_derivatives[:, :2]
predicted_derivatives_xy = predicted_derivatives[:, :2]

# Compute the R^2 score for the x and y derivatives
r2 = r2_score(true_derivatives_xy, predicted_derivatives_xy)

print(f"Overall R^2 score for x and y derivatives (rounded): {r2}")

(x)' = 0.199 x + 0.962 y + 0.192 y mu + -0.963 x^3 + -0.954 x y^2
(y)' = -0.963 x + 0.200 y + -0.193 x mu + -0.948 x^2 y + -0.973 y^3
(mu)' = 0.000
Overall R^2 score for x and y derivatives (rounded): 0.9999864775032992


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pysindy/optimizers/stlsq.py:201: UserWarning: Sparsity parameter is too big (0.1) and eliminated all coefficients
  warnings.warn(
